In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

from difflib import SequenceMatcher
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import psycopg2
from statsmodels.stats.proportion import proportion_confint

In [ ]:
# Establish a connection to the database
conn = psycopg2.connect(
    host="",  # e.g., "localhost"
    database="",
    user="",
    password=""
)

# SQL query to retrieve the relevant data for both Rule 1 and Rule 2, excluding Unknown labels
query = """
WITH rule1_labeled AS (
    -- Label Rule 1 predictions
    SELECT l.empi, 
           CASE 
               WHEN l.final_label = 'Yes' THEN 1
               WHEN l.final_label = 'No' THEN 0
               ELSE NULL
           END AS final_label,
           CASE WHEN r1.empi IS NOT NULL THEN 1 ELSE 0 END AS rule1_label
    FROM adrd.adrd_study_700_label l
    LEFT JOIN adrd.dementia_cases_rule1 r1 ON l.empi = r1.empi
    WHERE l.final_label != 'Unknown' and l.rand_ind > 33
),
rule2_labeled AS (
    -- Label Rule 2 predictions
    SELECT l.empi, 
           CASE 
               WHEN l.final_label = 'Yes' THEN 1
               WHEN l.final_label = 'No' THEN 0
               ELSE NULL
           END AS final_label,
           CASE WHEN r2.empi IS NOT NULL THEN 1 ELSE 0 END AS rule2_label
    FROM adrd.adrd_study_700_label l
    LEFT JOIN adrd.dementia_cases_rule2 r2 ON l.empi = r2.empi
    WHERE l.final_label != 'Unknown' and l.rand_ind > 33
)
-- Combine Rule 1 and Rule 2 predictions with final labels
SELECT r1.empi, r1.final_label, r1.rule1_label, r2.rule2_label
FROM rule1_labeled r1
JOIN rule2_labeled r2 ON r1.empi = r2.empi;
"""


# Execute the query and load the results into a pandas DataFrame
data = pd.read_sql_query(query, conn)

# Close the database connection
conn.close()

# Check if any data was returned
if not data.empty:
    def calculate_metrics(rule_label, final_label):
        # Calculate confusion matrix components
        tn, fp, fn, tp = confusion_matrix(final_label, rule_label).ravel()

        # Calculate the total population
        total_population = len(final_label)

        # Calculate performance metrics
        sensitivity = tp / (tp + fn) if tp + fn != 0 else 0
        specificity = tn / (tn + fp) if tn + fp != 0 else 0
        ppv = tp / (tp + fp) if tp + fp != 0 else 0
        npv = tn / (tn + fn) if tn + fn != 0 else 0
        f1 = f1_score(final_label, rule_label)

        # Calculate 95% confidence intervals for sensitivity, specificity, PPV, NPV
        sensitivity_ci = proportion_confint(tp, tp + fn, method='wilson')
        specificity_ci = proportion_confint(tn, tn + fp, method='wilson')
        ppv_ci = proportion_confint(tp, tp + fp, method='wilson')
        npv_ci = proportion_confint(tn, tn + fn, method='wilson')

        # Return the calculated metrics as a dictionary
        return {
            "Total population": total_population,
            "True positive (TP)": tp,
            "False negative (FN)": fn,
            "True negative (TN)": tn,
            "False positive (FP)": fp,
            "Sensitivity": (sensitivity, sensitivity_ci),
            "Specificity": (specificity, specificity_ci),
            "PPV": (ppv, ppv_ci),
            "NPV": (npv, npv_ci),
            "F1 Score": f1
        }

    # Calculate metrics for Rule 1
    rule1_metrics = calculate_metrics(data['rule1_label'], data['final_label'])

    # Calculate metrics for Rule 2
    rule2_metrics = calculate_metrics(data['rule2_label'], data['final_label'])

    # Function to print the detailed results
    def print_metrics(rule_name, metrics):
        print(f"{rule_name} Metrics")
        print(f"Total population: {metrics['Total population']}")
        print(f"True positive (TP): {metrics['True positive (TP)']}")
        print(f"False negative (FN): {metrics['False negative (FN)']}")
        print(f"True negative (TN): {metrics['True negative (TN)']}")
        print(f"False positive (FP): {metrics['False positive (FP)']}")

        sensitivity, sensitivity_ci = metrics['Sensitivity']
        specificity, specificity_ci = metrics['Specificity']
        ppv, ppv_ci = metrics['PPV']
        npv, npv_ci = metrics['NPV']
        f1 = metrics['F1 Score']

        print(f"Sensitivity: {sensitivity:.2%} (95% CI: {sensitivity_ci[0]:.2%} - {sensitivity_ci[1]:.2%})")
        print(f"Specificity: {specificity:.2%} (95% CI: {specificity_ci[0]:.2%} - {specificity_ci[1]:.2%})")
        print(f"Positive Predictive Value (PPV): {ppv:.2%} (95% CI: {ppv_ci[0]:.2%} - {ppv_ci[1]:.2%})")
        print(f"Negative Predictive Value (NPV): {npv:.2%} (95% CI: {npv_ci[0]:.2%} - {npv_ci[1]:.2%})")
        print(f"F1 Score: {f1:.2f}")
        print()

    # Print results for Rule 1
    print_metrics("Rule 1", rule1_metrics)

    # Print results for Rule 2
    print_metrics("Rule 2", rule2_metrics)

else:
    print("No data available for processing after filtering out 'Unknown' labels.")
        